In [1]:
import json
import os

os.chdir("..")

In [ ]:
from pathlib import Path
from collections import defaultdict

# mns = [1,2, 13, 15, 14, 6, 9, 4, 5, 7, 12] # [13,15,14, 9, 6, 5, 7, 12, 10, 3, 4, 11, 8]]
mns = [1,2, 13, 15, 14, 6, 9]

cur_dir = Path(".").absolute()

def load_dataset_from_file(domain_name, task_name, model_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/{model_name}/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)["instances"]
    
    
eval_results_0 = defaultdict(dict)
eval_results_1 = defaultdict(dict)

for i in mns:
    for t in range(3):    
        eval_results_0[i][t] = {
            x["dataset_idx"]: x["llm_correct"] for x in 
            load_dataset_from_file(f"blocksworld_mystery_{i}", "plan_generation_po", f"qwq-32b-steered-full-0-il-60-mean-triple-8-greedy_{t}")
        }
        
        eval_results_1[i][t] = {
            x["dataset_idx"]: x["llm_correct"] for x in 
            load_dataset_from_file(f"blocksworld_mystery_{i}", "plan_generation_po", f"qwq-32b-steered-full-1-il-60-mean-triple-8-greedy_{t}")
        }

In [49]:
eval_results_0_sum = {}
eval_results_1_sum = {}

for i in mns:
    eval_results_0_sum[i] = defaultdict(int)
    eval_results_1_sum[i] = defaultdict(int)
    
    for t in range(3):
        for k, v in eval_results_0[i][t].items():
            eval_results_0_sum[i][k] += v
        for k, v in eval_results_1[i][t].items():
            eval_results_1_sum[i][k] += v
        
    for k in range(100):
        eval_results_0_sum[i][k] = eval_results_0_sum[i][k] >= 2
        eval_results_1_sum[i][k] = eval_results_1_sum[i][k] >= 2

In [50]:
deltas = defaultdict(dict)

for i in mns:
    for k in eval_results_0_sum[i].keys():
        deltas[i][k] = eval_results_1_sum[i][k] - eval_results_0_sum[i][k]

In [51]:

for i in mns:
    print(f"{i}: {sum(eval_results_1_sum[i].values())} - {sum(eval_results_0_sum[i].values())} = {sum(deltas[i].values())}")

1: 30 - 33 = -3
2: 38 - 41 = -3
13: 35 - 35 = 0


In [44]:
print("Average improvement: ", sum([sum(deltas[i].values()) for i in mns]) / len(mns))

Average improvement:  -0.7142857142857143


In [14]:
eval_results = [
    [x["llm_correct"] for x in y] for y in eval_results
]

In [16]:
correct_ids = [
    [i for i, x in enumerate(y) if x] for y in eval_results
]

In [ ]:
set(correct_ids[1]) - set(correct_ids[0])

{1, 4, 10, 34, 40, 41, 46, 47, 50, 54, 57, 66, 81, 84, 90, 92, 95}

: 

In [5]:
data[0]["steered_generation"] == data_2[0]["steered_generation"]

False

In [8]:
print(data[1]["steered_generation"])

 (Wait, but Nexus is a global fact, so this would set the global Nexus to true?)

Wait, that's a critical point. Let me recheck the problem's action descriptions:

The user wrote:

"For Silence action, the following needs to be true: Pulse object.

Once Silence action is performed the following will be true: Essence object, Aura object, Nexus. 

Once Silence action is performed the following will be false: Pulse object."

Ah, so the Silence action on an object (say, object X) will set the following:

- Essence X becomes true,

- Aura X becomes true,

- The global Nexus becomes true,

and Pulse X becomes false.

Wait, that's a bit odd because the global Nexus is set to true regardless of the object. So if you Silence any object with Pulse, you can restore the global Nexus. That's important.

So in the scenario where after Divesting D from A, we have lost Nexus. To get it back, we can Silence D (since D has Pulse). 

So after Divesting D from A:

- Essence A is true,

- Pulse D is true,


In [9]:
print(data_2[1]["steered_generation"])

 (Wait, but Nexus is a global fact, so this would set the global Nexus to true?)

Wait, that's a bit confusing. Let me recheck the problem's action descriptions:

The user provided:

For Silence action:

"Once Silence action is performed the following will be true: Essence object, Aura object, Nexus. Once Silence action is performed the following will be false: Pulse object."

So, the Silence action on an object (say, object X) will set:

- Essence X becomes true,

- Aura X becomes true,

- Nexus (global) becomes true,

and Pulse X becomes false.

Wait, that's a bit odd because the global Nexus is set to true regardless of the object. So if you Silence any object with Pulse, it will set the global Nexus to true again. That's important.

So going back to the Divest D from A scenario:

After Divesting D from A, we have:

- Essence A is true,

- Pulse D is true,

- Essence D is false,

- Nexus is false (because Divest's effect removes Nexus),

- D no longer harmonizes A.

Then, if we Sile

In [19]:
eqs = [data[i]["original_input"] == data_2[i]["original_input"] for i in range(len(data))]

In [ ]:
sum(eqs)

50

: 